In [1]:
import os
print(os.getcwd())
print(os.listdir())

/home/kieutrang/Documents/Project 2
['id_info.ipynb', 'data', 'products-0-200000.xlsx', '.venv']


## Checking the logicality of data set ID information 

### Load data set and see the information 

In [12]:
import pandas as pd

df = pd.read_excel("products-0-200000.xlsx")

print("Rows:", len(df))
print("Columns:", df.columns.tolist())
print("Empty ids:", df["id"].isna().sum())
print("Duplicate ids:", len(df) - df["id"].nunique())
print("Smallest id:", df["id"].min())
print("Largest id:", df["id"].max())

Rows: 200000
Columns: ['id']
Empty ids: 0
Duplicate ids: 0
Smallest id: 54380
Largest id: 263677051


## Calling API for sample test 

In [13]:
from curl_cffi import requests

URL = "https://api.tiki.vn/product-detail/api/v1/products/"

r = requests.get(URL + "1391347", impersonate="chrome", timeout=30)
print("Status:", r.status_code)
print("Content-Type:", r.headers.get("content-type"))

data = r.json()
print("Number of fields:", len(data))

Status: 200
Content-Type: application/json; charset=utf-8
Number of fields: 59


In [14]:
# Look at the 6 fields we need
print("id:", data["id"])
print("name:", data["name"])
print("url_key:", data["url_key"])
print("price:", data["price"])
print("description:", data["description"][:300])
print("images[0]:", data["images"][0])

id: 1391347
name: Tranh xếp Hình Tia Sáng  Gương Thu Tia Sáng (2035 Mảnh Ghép)
url_key: tranh-xep-hinh-tia-sang-guong-thu-tia-sang-2035-manh-ghep-p1391347
price: 211200
description: <ul>
<li><strong>Bộ Xếp Hình Gương Thu Tia Sáng (2035 Mảnh Ghép) </strong>được làm từ chất liệu giấy bồi cao cấp, bền đẹp không gây độc hại an toàn cho sức khỏe của bé.</li>
<li>Với màu sắc sinh động cùng ý tưởng về gương thu sẽ mang đến cho trẻ những giây phút trải nghiệm độc đáo, thú vị.</li>
<li>
images[0]: {'base_url': 'https://salt.tikicdn.com/ts/product/f4/6e/ff/18952da934d8ae12e0bd1edcf0ac2c8f.jpg', 'is_gallery': True, 'label': None, 'large_url': 'https://salt.tikicdn.com/cache/w1200/ts/product/f4/6e/ff/18952da934d8ae12e0bd1edcf0ac2c8f.jpg', 'medium_url': 'https://salt.tikicdn.com/cache/w300/ts/product/f4/6e/ff/18952da934d8ae12e0bd1edcf0ac2c8f.jpg', 'position': None, 'small_url': 'https://salt.tikicdn.com/cache/200x280/ts/product/f4/6e/ff/18952da934d8ae12e0bd1edcf0ac2c8f.jpg', 'thumbnail_url': 'https

## How fast can we call the API?
Test A: 20 requests with no pause. 
Test B: 20 other requests with a 1 second pause.
Each line shows: id, status code, Content-Type.
- `application/json` = real product data
- `404` = the product does not exist
- `200` + `text/html` = **blocked**: Tiki sends a web page instead of data

In [16]:
# Test A: no pause
import time

session = requests.Session(impersonate="chrome")

for pid in df["id"].tolist()[0:20]:
    r = session.get(URL + str(pid), timeout=30)
    print(pid, r.status_code, r.headers.get("content-type"))

1391347 200 application/json; charset=utf-8
74897599 200 application/json; charset=utf-8
154155413 200 application/json; charset=utf-8
253117062 200 application/json; charset=utf-8
130978358 200 application/json; charset=utf-8
214009046 200 application/json; charset=utf-8
171618108 200 application/json; charset=utf-8
179970479 200 application/json; charset=utf-8
139457837 200 application/json; charset=utf-8
197334787 200 application/json; charset=utf-8
214008432 200 text/html
213678661 200 text/html
138083218 200 text/html
75331097 200 text/html
139457787 200 text/html
146457002 200 text/html
178596691 200 text/html
191463903 200 text/html
167750518 200 text/html
170337343 200 text/html


In [17]:
# Test B: 1 second pause after each request
for pid in df["id"].tolist()[20:40]:
    r = session.get(URL + str(pid), timeout=30)
    print(pid, r.status_code, r.headers.get("content-type"))
    time.sleep(1)

58510250 200 text/html
214006984 200 text/html
75378756 200 text/html
215101707 200 text/html
184319033 200 text/html
205930779 200 text/html
152738111 200 text/html
103637219 200 text/html
67029245 200 text/html
154530234 200 text/html
157943579 200 text/html
168663152 200 text/html
184669779 200 text/html
160814241 200 text/html
225385243 200 text/html
36376636 200 text/html
140420532 200 text/html
206303845 200 text/html
138838287 200 text/html
74892021 200 text/html


## Checking the result of crawling data

In [2]:
import json
from pathlib import Path

P, E = [], []
for i in range(1, 201):
    P += json.loads(Path(f"data/products/products_{i:03d}.json").read_text(encoding="utf-8"))
    E += json.loads(Path(f"data/errors/errors_{i:03d}.json").read_text(encoding="utf-8"))

print(len(P), "products,", len(E), "errors, total =", len(P) + len(E))

124299 products, 75701 errors, total = 200000


In [7]:
import pandas as pd

list_ids = set(pd.read_excel("products-0-200000.xlsx")["id"])
crawl_ids = [p["id"] for p in P] + [e["id"] for e in E]

print("Number of product:", len(list_ids))
print("Duplicate id:", len(crawl_ids) - len(set(crawl_ids)))
print("Under the list:", len(list_ids - set(crawl_ids)))
print("Over the list:", len(set(crawl_ids) - list_ids))

Number of product: 200000
Duplicate id: 0
Under the list: 0
Over the list: 0


In [6]:
from collections import Counter

Counter(e["reason"] for e in E)

Counter({'404 not found': 75701})

In [11]:
print(list(Path("data").rglob("*.tmp")))

[]
